# Dep Parsing using word nodes only.

In [7]:
from src.multi_graph.data_preprocessing import preprocessing_legal_pt
# main.py

# This script demonstrates how to convert a text's dependency parse into a network graph
# and visualize it interactively.
#
# Required libraries:
# - spacy: For natural language processing and dependency parsing.
# - networkx: For creating and managing the graph structure.
# - pyvis: For creating interactive HTML visualizations of the graph.
# - A spacy model: We'll use 'en_core_web_sm'.
#
# You can install them using pip:
# pip install spacy networkx pyvis
# python -m spacy download en_core_web_sm

import spacy
import networkx as nx
from pyvis.network import Network
import webbrowser
import os

def text_to_dependency_graph(text):
    """
    Parses text using spacy, builds a networkx graph from dependencies,
    and returns the graph.

    Args:
        text (str): The input text to process.

    Returns:
        nx.Graph: A NetworkX graph representing the dependency structure.
    """
    # Load the pre-trained English model for spacy
    # This model contains the components for parsing and dependency analysis.
    try:
        nlp = spacy.load("pt_core_news_lg")
    except OSError:
        print("Spacy model 'pt_core_news_lg' not found.")
        print("Please run: python -m spacy download en_core_web_sm")
        return None

    text = preprocessing_legal_pt(
        text=text,
        nlp_spacy=nlp
    )

    # Process the text with the spacy pipeline
    doc = nlp(text)

    # Create a new directed graph
    G = nx.DiGraph()

    # Add nodes and edges for each token and its dependency
    for token in doc:
        # Add nodes for the token (head) and its child
        # We use the token's index `i` as a unique ID for the node
        G.add_node(token.i, label=token.text, title=f"POS: {token.pos_}, Tag: {token.tag_}")

        # The head attribute points to the parent token in the dependency tree.
        # We add a directed edge from the head to the token.
        if token.head is not None and token.head.i != token.i:
            G.add_edge(token.head.i, token.i, label=token.dep_)

    return G

def visualize_graph(graph, filename="dependency_graph.html"):
    """
    Converts a networkx graph to a pyvis network and saves it as an
    interactive HTML file.

    Args:
        graph (nx.Graph): The networkx graph to visualize.
        filename (str): The name of the output HTML file.
    """
    if graph is None:
        print("Graph is None. Cannot visualize.")
        return

    # Create a pyvis network object
    # We specify the size, directed nature, and notebook environment for display.
    net = Network(height="800px", width="100%", bgcolor="#222222", font_color="white", directed=True)

    # Add nodes and edges from the networkx graph to the pyvis network
    net.from_nx(graph)

    # Customize the physics and interaction options for a better user experience
    net.set_options("""
    var options = {
      "nodes": {
        "shape": "dot",
        "size": 20,
        "font": {
          "size": 18
        },
        "borderWidth": 2,
        "shadow": true
      },
      "edges": {
        "width": 2,
        "shadow": true,
        "smooth": true,
        "arrows": {
          "to": { "enabled": true, "scaleFactor": 1 }
        },
        "font": {
            "size": 14,
            "align": "horizontal"
        }
      },
      "physics": {
        "enabled": true,
        "barnesHut": {
          "gravitationalConstant": -30000,
          "centralGravity": 0.3,
          "springLength": 200,
          "springConstant": 0.04
        },
        "solver": "barnesHut"
      },
      "interaction": {
        "dragNodes": true,
        "dragView": true,
        "hideEdgesOnDrag": false,
        "hideNodesOnDrag": false,
        "hover": true,
        "tooltipDelay": 300
      }
    }
    """)

    # Generate and save the HTML file
    try:
        net.save_graph(filename)
        print(f"Successfully generated interactive graph: {filename}")

        # Automatically open the generated file in the default web browser
        webbrowser.open('file://' + os.path.realpath(filename))

    except Exception as e:
        print(f"An error occurred while saving the graph: {e}")



In [8]:


# --- Main execution block ---
if __name__ == "__main__":
    # Example sentence to analyze. You can change this to any text you want.
    sample_text = open("../data/datasets/STF_HC/full/raw/Preso/costa001.txt").read()

    print(f"Analyzing text: '{sample_text}'")

    # 1. Convert the text to a dependency graph
    dependency_graph = text_to_dependency_graph(sample_text)

    # 2. Visualize the graph as an interactive HTML file
    if dependency_graph:
        visualize_graph(dependency_graph)

Analyzing text: '
                                O SENHOR MINISTRO MARCO AURÉLIO – O Gabinete assim
                        resumiu o quadro revelado neste processo:
                                                       O paciente foi denunciado pela suposta prática do crime
                                              descrito no artigo 157, § 2º, incisos I e II (roubo circunstanciado
                                              pelo uso de armas e concurso de pessoas), do Código Penal. Ao
                                              receber a denúncia, o Juízo da 1ª Vara Criminal de
                                              Sertãozinho/SP determinou a prisão preventiva, por
                                              conveniência da instrução criminal. Ressaltou não haver o
                                              paciente atendido às intimações da autoridade policial nem
                                              apresentado justificativa plausível para deixar de 

# Sentence nodes only.

In [9]:
# main.py

# This script demonstrates a two-level approach to visualizing text structure:
# 1. A detailed word-level Dependency Graph.
# 2. A high-level, abstracted Conceptual Phrase Graph.
#
# Required libraries:
# - spacy: For natural language processing and dependency parsing.
# - networkx: For creating and managing the graph structure.
# - pyvis: For creating interactive HTML visualizations of the graph.
# - A spacy model: We'll use 'en_core_web_sm'.
#
# You can install them using pip:
# pip install spacy networkx pyvis
# python -m spacy download en_core_web_sm

import spacy
import networkx as nx
from pyvis.network import Network
import webbrowser
import os
from itertools import combinations

def text_to_dependency_graph(doc):
    """
    Builds a detailed networkx graph from a text's token-level dependencies.
    As discussed, using the token's index `token.i` as the node ID ensures that
    repeated words are treated as distinct nodes.

    Args:
        doc (spacy.Doc): The processed spacy document.

    Returns:
        nx.DiGraph: A NetworkX graph representing the dependency structure.
    """
    # Create a new directed graph
    G = nx.DiGraph()

    # Add nodes and edges for each token and its dependency
    for token in doc:
        # Add a node for each token. The label is the token's text.
        # The title contains extra info (Part-of-Speech, Tag) for the hover tooltip.
        G.add_node(token.i, label=token.text, title=f"POS: {token.pos_}, Tag: {token.tag_}")

        # Add a directed edge from the token's head (parent) to the token itself.
        # The edge is labeled with the dependency type.
        if token.head is not None and token.head.i != token.i:
            G.add_edge(token.head.i, token.i, label=token.dep_)

    return G

def phrases_to_conceptual_graph(doc, dep_graph):
    """
    Builds a high-level conceptual graph where nodes are noun phrases (chunks)
    and edges represent the actions or relationships connecting them.

    Args:
        doc (spacy.Doc): The processed spacy document.
        dep_graph (nx.DiGraph): The token-level dependency graph.

    Returns:
        nx.DiGraph: A NetworkX graph representing the conceptual phrase structure.
    """
    conceptual_G = nx.DiGraph()

    # Use spacy's built-in noun chunk identification
    noun_chunks = list(doc.noun_chunks)

    # Add a node for each identified noun chunk
    for chunk in noun_chunks:
        conceptual_G.add_node(chunk.text, label=chunk.text, title=f"Root: {chunk.root.text}")

    # Find relationships between every pair of noun chunks
    for chunk1, chunk2 in combinations(noun_chunks, 2):
        # The "root" of a chunk is its main noun, which connects it to the rest of the sentence.
        root1 = chunk1.root
        root2 = chunk2.root

        try:
            # Find the shortest path in the dependency graph between the roots of the two chunks.
            # This path represents the grammatical connection.
            path_indices = nx.shortest_path(dep_graph.to_undirected(), source=root1.i, target=root2.i)

            # Find the common ancestor (head) in the dependency tree. This is often the main verb.
            path_tokens = [doc[i] for i in path_indices]
            lca = doc[nx.lowest_common_ancestor(dep_graph, root1.i, root2.i)]

            # Reconstruct the connecting phrase from the path, excluding the chunks themselves.
            # We filter for verbs, adpositions (like 'over', 'with'), and agents.
            edge_label_parts = []
            for token in path_tokens:
                if token not in chunk1 and token not in chunk2:
                     if token.pos_ in ('VERB', 'ADP', 'AGENT'):
                        edge_label_parts.append(token.text)

            # If a connecting verb/preposition is found, create an edge.
            if edge_label_parts:
                edge_label = " ".join(edge_label_parts)
                # The Lowest Common Ancestor (often the verb) determines the direction.
                if lca == root1.head: # root1 is object of verb
                    conceptual_G.add_edge(chunk2.text, chunk1.text, label=edge_label)
                else: # root1 is subject of verb
                    conceptual_G.add_edge(chunk1.text, chunk2.text, label=edge_label)

        except (nx.NetworkXNoPath, nx.NodeNotFound):
            # No path exists between these chunks in the dependency graph.
            continue

    return conceptual_G


def visualize_graph(graph, filename, is_conceptual=False):
    """
    Converts a networkx graph to a pyvis network and saves it as an
    interactive HTML file.

    Args:
        graph (nx.Graph): The networkx graph to visualize.
        filename (str): The name of the output HTML file.
        is_conceptual (bool): Flag to apply different styling for the conceptual graph.
    """
    if graph is None or not graph.nodes:
        print(f"Graph for {filename} is empty or None. Skipping visualization.")
        return

    net = Network(height="800px", width="100%", bgcolor="#222222", font_color="white", directed=True)
    net.from_nx(graph)

    # Apply different styling for the more abstract conceptual graph
    if is_conceptual:
        options = """
        var options = {
          "nodes": {
            "shape": "box",
            "size": 30,
            "font": { "size": 22, "color": "#ffffff" },
            "borderWidth": 3,
            "shadow": true
          },
          "edges": {
            "width": 3,
            "shadow": true,
            "smooth": { "type": "dynamic" },
            "arrows": { "to": { "enabled": true, "scaleFactor": 1.5 } },
            "font": { "size": 18, "align": "middle", "strokeWidth": 5, "strokeColor": "#222222" }
          },
          "physics": { "enabled": true, "solver": "repulsion", "repulsion": { "nodeDistance": 250 } },
          "interaction": { "dragNodes": true, "dragView": true, "zoomView": true, "hover": true }
        }
        """
    else: # Default options for dependency graph
        options = """
        var options = {
          "nodes": { "shape": "dot", "size": 20, "font": { "size": 18 } },
          "edges": { "width": 2, "shadow": true, "smooth": true, "font": { "size": 14, "align": "horizontal" } },
          "physics": { "enabled": true, "barnesHut": { "gravitationalConstant": -30000, "springLength": 200 } },
          "interaction": { "dragNodes": true, "dragView": true, "hover": true }
        }
        """

    net.set_options(options)

    try:
        net.save_graph(filename)
        print(f"Successfully generated interactive graph: {filename}")
        webbrowser.open('file://' + os.path.realpath(filename))
    except Exception as e:
        print(f"An error occurred while saving the graph: {e}")



In [10]:

# --- Main execution block ---
if __name__ == "__main__":
    # Example sentence to analyze. Try changing it!
    sample_text = open("../data/datasets/STF_HC/full/raw/Preso/costa001.txt").read()

    print(f"Analyzing text: '{sample_text}'")

    # Load the spacy model once
    try:
        nlp = spacy.load("pt_core_news_lg")
    except OSError:
        print("Spacy model 'pt_core_news_lg' not found.")
        print("Please run: python -m spacy download en_core_web_sm")
        exit()
    sample_text = preprocessing_legal_pt(
        text=sample_text,
        nlp_spacy=nlp
    )

    # Process the text with the spacy pipeline
    doc = nlp(sample_text)

    # --- 1. Generate and Visualize the Detailed Dependency Graph ---
    print("\n--- Generating Dependency Graph ---")
    dependency_graph = text_to_dependency_graph(doc)
    visualize_graph(dependency_graph, "dependency_graph.html")

    # --- 2. Generate and Visualize the High-Level Conceptual Graph ---
    print("\n--- Generating Conceptual Phrase Graph ---")
    conceptual_graph = phrases_to_conceptual_graph(doc, dependency_graph)
    visualize_graph(conceptual_graph, "conceptual_graph.html", is_conceptual=True)



Analyzing text: '
                                O SENHOR MINISTRO MARCO AURÉLIO – O Gabinete assim
                        resumiu o quadro revelado neste processo:
                                                       O paciente foi denunciado pela suposta prática do crime
                                              descrito no artigo 157, § 2º, incisos I e II (roubo circunstanciado
                                              pelo uso de armas e concurso de pessoas), do Código Penal. Ao
                                              receber a denúncia, o Juízo da 1ª Vara Criminal de
                                              Sertãozinho/SP determinou a prisão preventiva, por
                                              conveniência da instrução criminal. Ressaltou não haver o
                                              paciente atendido às intimações da autoridade policial nem
                                              apresentado justificativa plausível para deixar de 

In [11]:
# main.py

# This script demonstrates a two-level approach to visualizing text structure:
# 1. A detailed word-level Dependency Graph.
# 2. A high-level, abstracted Conceptual Phrase Graph with improved connectivity.
#
# Required libraries:
# - spacy: For natural language processing and dependency parsing.
# - networkx: For creating and managing the graph structure.
# - pyvis: For creating interactive HTML visualizations of the graph.
# - A spacy model: We'll use 'en_core_web_sm'.
#
# You can install them using pip:
# pip install spacy networkx pyvis
# python -m spacy download en_core_web_sm

import spacy
import networkx as nx
from pyvis.network import Network
import webbrowser
import os

def text_to_dependency_graph(doc):
    """
    Builds a detailed networkx graph from a text's token-level dependencies.
    As discussed, using the token's index `token.i` as the node ID ensures that
    repeated words are treated as distinct nodes.

    Args:
        doc (spacy.Doc): The processed spacy document.

    Returns:
        nx.DiGraph: A NetworkX graph representing the dependency structure.
    """
    G = nx.DiGraph()
    for token in doc:
        G.add_node(token.i, label=token.text, title=f"POS: {token.pos_}, Tag: {token.tag_}")
        if token.head is not None and token.head.i != token.i:
            G.add_edge(token.head.i, token.i, label=token.dep_)
    return G

def phrases_to_conceptual_graph(doc):
    """
    Builds a high-level conceptual graph where nodes are noun phrases (chunks)
    and edges represent the actions or relationships connecting them. This version
    uses a more robust method to connect phrases and avoid "islands".

    Args:
        doc (spacy.Doc): The processed spacy document.

    Returns:
        nx.DiGraph: A NetworkX graph representing the conceptual phrase structure.
    """
    conceptual_G = nx.DiGraph()

    # Use spacy's built-in noun chunk identification
    noun_chunks = list(doc.noun_chunks)

    # Add a node for each identified noun chunk
    for chunk in noun_chunks:
        conceptual_G.add_node(chunk.text, label=chunk.text, title=f"Root: {chunk.root.text}")

    # Find relationships by iterating through all tokens, focusing on verbs
    for token in doc:
        # We are interested in verbs as they often define the main action or relationship
        if token.pos_ == 'VERB':
            subjects = []
            objects = []

            # Find subjects and objects connected to this verb
            for child in token.children:
                # nsubj: nominal subject, nsubjpass: nominal subject (passive)
                if child.dep_ in ('nsubj', 'nsubjpass'):
                    # Find the noun chunk that this subject token belongs to
                    for chunk in noun_chunks:
                        if child in chunk:
                            subjects.append(chunk.text)
                            break
                # dobj: direct object, pobj: object of preposition
                elif child.dep_ in ('dobj', 'pobj'):
                    for chunk in noun_chunks:
                        if child in chunk:
                            objects.append(chunk.text)
                            break

            # If we have at least one subject and one object, we can form a relationship
            if subjects and objects:
                # The edge label is the verb itself
                edge_label = token.text

                # If the verb has a preposition attached, add it to the label
                # e.g., in "jumps over", 'over' is a prepositional child of 'jumps'
                for child in token.children:
                    if child.dep_ == 'prep':
                        edge_label += f" {child.text}"

                # Create an edge from every subject to every object
                for subj in subjects:
                    for obj in objects:
                        conceptual_G.add_edge(subj, obj, label=edge_label)

    # Second pass for adjectival and prepositional relationships (e.g., "fox with a tail")
    for chunk in noun_chunks:
        # The root of a chunk is its core noun. Its head is what it connects to.
        if chunk.root.head.pos_ not in ('VERB'):
             # Find the noun chunk of the head token
             for head_chunk in noun_chunks:
                 if chunk.root.head in head_chunk and chunk != head_chunk:
                     # The relationship is the preposition or connecting word
                     edge_label = chunk.root.head.text
                     conceptual_G.add_edge(head_chunk.text, chunk.text, label=edge_label)
                     break

    return conceptual_G


def visualize_graph(graph, filename, is_conceptual=False):
    """
    Converts a networkx graph to a pyvis network and saves it as an
    interactive HTML file.

    Args:
        graph (nx.Graph): The networkx graph to visualize.
        filename (str): The name of the output HTML file.
        is_conceptual (bool): Flag to apply different styling for the conceptual graph.
    """
    if graph is None or not graph.nodes:
        print(f"Graph for {filename} is empty or None. Skipping visualization.")
        return

    net = Network(height="800px", width="100%", bgcolor="#222222", font_color="white", directed=True)
    net.from_nx(graph)

    # Apply different styling for the more abstract conceptual graph
    if is_conceptual:
        options = """
        var options = {
          "nodes": {
            "shape": "box",
            "size": 30,
            "font": { "size": 22, "color": "#ffffff" },
            "borderWidth": 3,
            "shadow": true
          },
          "edges": {
            "width": 3,
            "shadow": true,
            "smooth": { "type": "dynamic" },
            "arrows": { "to": { "enabled": true, "scaleFactor": 1.5 } },
            "font": { "size": 18, "align": "middle", "strokeWidth": 5, "strokeColor": "#222222" }
          },
          "physics": { "enabled": true, "solver": "repulsion", "repulsion": { "nodeDistance": 300 } },
          "interaction": { "dragNodes": true, "dragView": true, "zoomView": true, "hover": true }
        }
        """
    else: # Default options for dependency graph
        options = """
        var options = {
          "nodes": { "shape": "dot", "size": 20, "font": { "size": 18 } },
          "edges": { "width": 2, "shadow": true, "smooth": true, "font": { "size": 14, "align": "horizontal" } },
          "physics": { "enabled": true, "barnesHut": { "gravitationalConstant": -30000, "springLength": 200 } },
          "interaction": { "dragNodes": true, "dragView": true, "hover": true }
        }
        """

    net.set_options(options)

    try:
        net.save_graph(filename)
        print(f"Successfully generated interactive graph: {filename}")
        webbrowser.open('file://' + os.path.realpath(filename))
    except Exception as e:
        print(f"An error occurred while saving the graph: {e}")


# --- Main execution block ---
if __name__ == "__main__":
    # Example sentence to analyze. Try changing it!
    sample_text = open("../data/datasets/STF_HC/full/raw/Preso/costa001.txt").read()

    print(f"Analyzing text: '{sample_text}'")

    # Load the spacy model once
    try:
        nlp = spacy.load("pt_core_news_lg")
    except OSError:
        print("Spacy model 'pt_core_news_lg' not found.")
        print("Please run: python -m spacy download en_core_web_sm")
        exit()
    sample_text = preprocessing_legal_pt(
        text=sample_text,
        nlp_spacy=nlp
    )

    # Process the text with the spacy pipeline
    doc = nlp(sample_text)

    # --- 1. Generate and Visualize the Detailed Dependency Graph ---
    print("\n--- Generating Dependency Graph ---")
    dependency_graph = text_to_dependency_graph(doc)
    visualize_graph(dependency_graph, "dependency_graph.html")

    # --- 2. Generate and Visualize the High-Level Conceptual Graph ---
    print("\n--- Generating Conceptual Phrase Graph ---")
    conceptual_graph = phrases_to_conceptual_graph(doc)
    visualize_graph(conceptual_graph, "conceptual_graph.html", is_conceptual=True)


Analyzing text: '
                                O SENHOR MINISTRO MARCO AURÉLIO – O Gabinete assim
                        resumiu o quadro revelado neste processo:
                                                       O paciente foi denunciado pela suposta prática do crime
                                              descrito no artigo 157, § 2º, incisos I e II (roubo circunstanciado
                                              pelo uso de armas e concurso de pessoas), do Código Penal. Ao
                                              receber a denúncia, o Juízo da 1ª Vara Criminal de
                                              Sertãozinho/SP determinou a prisão preventiva, por
                                              conveniência da instrução criminal. Ressaltou não haver o
                                              paciente atendido às intimações da autoridade policial nem
                                              apresentado justificativa plausível para deixar de 